In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json
import torch.optim as optim

In [ ]:
class ChessDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, max_len=200):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=1024)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x shape: [batch, seq_len] → transformer expects [seq_len, batch]
        x = x.transpose(0, 1)
        seq_len, batch_size = x.size()

        # Add embeddings
        positions = torch.arange(seq_len, device=x.device).unsqueeze(1)
        x = self.embed(x) + self.pos_embed(positions)

        # Decoder masking: prevent attention to future tokens
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()

        x = self.decoder(x, x, tgt_mask=mask)
        logits = self.fc_out(x)  # [seq_len, batch, vocab_size]
        return logits.transpose(0, 1)  # [batch, seq_len, vocab_size]


In [10]:
# Load your encoded games
encoded_tensor = torch.load("encoded_games_small.pt")

print(type(encoded_tensor))
print(encoded_tensor.shape)
print(encoded_tensor[0][:10])  # first few move IDs of first game


<class 'torch.Tensor'>
torch.Size([49854, 200])
tensor([    1, 10536, 10644, 10609, 10542, 10683,   303,  1791,  1347,  1338])


In [11]:
class ChessDataset(Dataset):
    def __init__(self, encoded_tensor):
        self.data = encoded_tensor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx][:-1]  # all but last
        y = self.data[idx][1:]   # all but first
        return x, y

dataset = ChessDataset(encoded_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)


In [15]:
with open("move_to_id.json", "r") as f:
    move_to_id = json.load(f)

vocab_size = len(move_to_id)
model = ChessDecoder(vocab_size=vocab_size)

In [16]:
x, y = next(iter(loader))

print("Input batch shape:", x.shape)
print("Target batch shape:", y.shape)
print("Example input sequence:", x[0][:10])
print("Example target sequence:", y[0][:10])

Input batch shape: torch.Size([64, 199])
Target batch shape: torch.Size([64, 199])
Example input sequence: tensor([    1, 10641, 10647,  1791, 10436, 10536, 10499,  2214,  1347,   123])
Example target sequence: tensor([10641, 10647,  1791, 10436, 10536, 10499,  2214,  1347,   123,    42])


In [17]:
with torch.no_grad():
    logits = model(x)
print("Model output shape:", logits.shape)

Model output shape: torch.Size([64, 199, 11017])


In [18]:
PAD_ID = move_to_id['<PAD>']
import torch.nn.functional as F

loss = F.cross_entropy(
    logits.reshape(-1, vocab_size),
    y.reshape(-1),
    ignore_index=PAD_ID
)
print("Test loss:", loss.item())


Test loss: 9.452414512634277


In [ ]:
vocab_size = len(move_to_id)

with open("best_hparams.json", "r") as f:
    best_hparams = json.load(f)

params = best_hparams["best_params"]

best_model = ChessDecoder(
    vocab_size=vocab_size,
    d_model=params["d_model"],
    nhead=params["nhead"],
    num_layers=params["num_layers"],
    max_len=encoded_tensor.size(1)
).to(device)

optimizer = optim.Adam(model.parameters(), lr=study.best_trial.params['lr'])

for epoch in range(3):
    for x, y in loader:
        logits = model(x)
        loss = F.cross_entropy(
            logits.reshape(-1, vocab_size),
            y.reshape(-1),
            ignore_index=PAD_ID
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} loss = {loss.items():.4f}")